# Exploratory Analysis: QSAR for Acetylcholinesterase Inhibitors

This notebook provides an overview of the dataset, feature engineering, and model interpretability for the AChE QSAR project.

**Author:** Semen Gavrilov

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
import joblib
import shap
import os

%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Load Data
We'll load the processed training and test sets.

In [ ]:
train_df = pd.read_csv('data/processed/train.csv')
test_df = pd.read_csv('data/processed/test.csv')
print(f'Train set size: {len(train_df)}')
print(f'Test set size: {len(test_df)}')
train_df.head()

## 2. Target Distribution (pIC50)
Let's visualize the distribution of biological activity.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(train_df['pIC50'], kde=True, color='skyblue')
plt.title('Distribution of pIC50 in Training Set')
plt.xlabel('pIC50')
plt.ylabel('Frequency')
plt.show()

## 3. Visualize Sample Molecules
Let's look at some highly active compounds.

In [ ]:
top_5 = train_df.nlargest(5, 'pIC50')
mols = [Chem.MolFromSmiles(s) for s in top_5['canonical_smiles']]
Draw.MolsToGridImage(mols, legends=[f'pIC50: {p:.2f}' for p in top_5['pIC50']], molsPerRow=5)

## 4. Model Interpretability (SHAP)
We'll load the trained XGBoost model and explain its predictions.

In [ ]:
reg_model = joblib.load('models/xgboost_regressor.joblib')
selected_features = joblib.load('models/selected_features.joblib')

X_test = test_df[selected_features]
explainer = shap.TreeExplainer(reg_model)
shap_values = explainer.shap_values(X_test)

# Summary Plot
shap.summary_plot(shap_values, X_test)